# 05 — GPU vs CPU benchmark (G5)

Formal same-host benchmark: pinned CPU threads, warmup excluded, CUDA event timing on GPU, separate compute-only and end-to-end (incl. host→device copy) numbers. The CPU side takes a while — that is the point.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q -e .


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.gpu.benchmark import run
from s2cd.train import build_model

cfg = load_config("configs/train.yaml")
model = build_model(cfg)
results = run(model, batch=16, channels=cfg["in_channels"], chip=256,
              iters=20, warmup=10)


In [ ]:
import json

rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
(rep / "benchmark.json").write_text(json.dumps(
    {**results, "device": describe_device(device)}, indent=1, default=str))
store.push_report("benchmark.json")
print("pushed benchmark.json -> Drive")
